<a href="https://colab.research.google.com/github/e-eki/rsna-pneumonia-yolo-pipeline/blob/development/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RSNA Pneumonia Detection — YOLO Pipeline (Colab)

End-to-end пайплайн: скачивание датасета → фильтрация разметки → обучение YOLO → оценка.

**Репозиторий:** https://github.com/e-eki/rsna-pneumonia-yolo-pipeline (ветка `development`)

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

**Файл `.ipynb` — только для Colab.** Kaggle-версия будет в отдельном ноутбуке (`kaggle_train.ipynb`).

## 0. Проверка GPU

In [1]:
!nvidia-smi

Wed Oct  7 09:26:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


## 1. Клонирование репозитория

In [3]:
%cd /content

# Если папка осталась от прошлой сессии — удаляем, чтобы клон был чистым.
!rm -rf rsna-pneumonia-yolo-pipeline

!git clone -b development https://github.com/e-eki/rsna-pneumonia-yolo-pipeline.git

/content
Cloning into 'rsna-pneumonia-yolo-pipeline'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 160 (delta 80), reused 105 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 997.83 KiB | 6.12 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [4]:
%cd rsna-pneumonia-yolo-pipeline
!git branch --show-current

/content/rsna-pneumonia-yolo-pipeline
development


## 2. Установка зависимостей

In [5]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 9.0 MB/s eta 0:00:00


## 3. Настройка Kaggle API через Colab Secrets

Токен хранится в Colab Secrets (🔑 на левой панели), **не в коде**.

- Имя секрета: `KAGGLE_API_TOKEN`
- Значение: `KGAT_...`

In [6]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ KAGGLE_API_TOKEN загружен из Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить KAGGLE_API_TOKEN. "
        "Добавьте секрет с этим именем в 🔑 на левой панели Colab."
    ) from e

# Проверка, что CLI видит токен
!kaggle datasets metadata poeticmage/rsna-pneumonia-detection-challenge-train-val

✅ KAGGLE_API_TOKEN загружен из Colab Secrets
Downloaded metadata to /content/rsna-pneumonia-yolo-pipeline/dataset-metadata.json


## 4. Скачивание датасета

In [7]:
!python -m src.download_data --config configs/config.yaml

2026-10-07 09:27:47 | INFO | __main__ | Скачивание датасета: poeticmage/rsna-pneumonia-detection-challenge-train-val
2026-10-07 09:27:47 | INFO | __main__ | Целевая директория:  /content/rsna-pneumonia-yolo-pipeline/data/raw
2026-10-07 09:31:34 | INFO | __main__ | Датасет скачан и распакован
2026-10-07 09:31:34 | INFO | __main__ | Готово. Данные в: data/raw


In [8]:
# Быстрая проверка структуры датасета
!echo "train images: $(ls data/raw/rsna_yolo/images/train | wc -l)"
!echo "train labels: $(ls data/raw/rsna_yolo/labels/train | wc -l)"
!echo "train labels с боксами: $(find data/raw/rsna_yolo/labels/train -name '*.txt' -size +0c | wc -l)"
!echo "val images:   $(ls data/raw/rsna_yolo/images/val | wc -l)"
!echo "val labels с боксами: $(find data/raw/rsna_yolo/labels/val -name '*.txt' -size +0c | wc -l)"

train images: 18678
train labels: 18678
train labels с боксами: 4208
val images:   8006
val labels с боксами: 1804


## 5. Подготовка датасета

Фильтрация «плохой» разметки, конвертация изображений, сборка `data/processed/` + `data.yaml`.

In [9]:
# Полная пересборка — сносим прошлые артефакты, чтобы результат был детерминированным.
!rm -rf data/processed data/staged data/samples
!python -m src.prepare_yolo_dataset --config configs/config.yaml

2026-10-07 09:31:46 | INFO | __main__ | Источник:             data/raw/rsna_yolo
2026-10-07 09:31:46 | INFO | __main__ | Назначение processed: data/processed
2026-10-07 09:31:46 | INFO | __main__ | Назначение rejected:  data/samples/rejected
2026-10-07 09:31:46 | INFO | __main__ | Формат вывода:        jpg (jpeg_quality=92)
2026-10-07 09:31:47 | INFO | __main__ | [train] Изображений: 18678 (output_format=jpg)
2026-10-07 09:36:07 | WARNING | __main__ | [train] 8da6abd4-8940-4696-bc43-6f0891cb7493: all_rejected=False, critical=True → rejected_images/
2026-10-07 09:40:20 | INFO | __main__ | [train] positive=4207, negative=14470, all_rejected=0, critical_rejected=1, dropped_to_rejected=1
2026-10-07 09:40:20 | INFO | __main__ | [train] boxes: total=6709, kept=6708, rejected=1, clipped=0
2026-10-07 09:40:20 | INFO | __main__ | [val] Изображений: 8006 (output_format=jpg)
2026-10-07 09:42:01 | WARNING | __main__ | [val] 847f25e9-6983-4c5f-a1e6-f3629557f1a3: all_rejected=False, critical=True → 

In [10]:
!ls data/processed
!ls data/processed/images/train | head -2
!ls data/processed/labels/train | head -2

data.yaml  images  labels
000924cf-0f8d-42bd-9158-1af53881a557.jpg
000fe35a-2649-43d4-b027-e67796d412e0.jpg
000924cf-0f8d-42bd-9158-1af53881a557.txt
000fe35a-2649-43d4-b027-e67796d412e0.txt


In [11]:
import pandas as pd
df = pd.read_csv("data/samples/rejected/rejected_annotations.csv")
print(df.to_string())

                              patientId  split          x           y       width  height    x_norm    y_norm    w_norm    h_norm          reject_reason  critical
0  8da6abd4-8940-4696-bc43-6f0891cb7493  train  592.00000  560.000512  320.000000   464.0  0.734375  0.773438  0.312500  0.453125  extends_out_of_bounds      True
1  847f25e9-6983-4c5f-a1e6-f3629557f1a3    val   75.00032  816.000512  307.999744   208.0  0.223633  0.898438  0.300781  0.203125  extends_out_of_bounds      True


## 6. Статистика и визуальные проверки

In [12]:
!python -m src.analyze_annotations --stage processed --split train
!python -m src.analyze_annotations --stage processed --split val

2026-10-07 09:58:52 | INFO | __main__ | Изображений: 18677, боксов: 6707
2026-10-07 09:58:52 | INFO | __main__ | Боксов: 6707
2026-10-07 09:58:53 | INFO | __main__ | ✅ Все YOLO-координаты в допустимых диапазонах
2026-10-07 09:58:53 | INFO | __main__ | Отчёт:   data/reports/processed_train/report.json
2026-10-07 09:58:53 | INFO | __main__ | Графики: data/reports/processed_train
2026-10-07 09:58:56 | INFO | __main__ | Изображений: 8005, боксов: 2844
2026-10-07 09:58:56 | INFO | __main__ | Боксов: 2844
2026-10-07 09:58:57 | INFO | __main__ | ✅ Все YOLO-координаты в допустимых диапазонах
2026-10-07 09:58:57 | INFO | __main__ | Отчёт:   data/reports/processed_val/report.json
2026-10-07 09:58:57 | INFO | __main__ | Графики: data/reports/processed_val


In [13]:
!python -m src.sample_and_draw --stage processed --split train --num 50 --grid
!python -m src.sample_and_draw --stage processed --split val   --num 50 --grid

2026-10-07 10:02:34 | INFO | __main__ | Собрано 18677 изображений из data/processed/images/train
2026-10-07 10:02:34 | INFO | __main__ | Сэмплировано 50 из 18677
2026-10-07 10:02:35 | INFO | __main__ | Отрисовано 50 изображений в data/samples/processed/train
2026-10-07 10:02:35 | INFO | __main__ | Метаданные: data/samples/processed/train/metadata.json
2026-10-07 10:02:35 | INFO | __main__ | Контактный лист: data/samples/processed/train/contact_sheet.jpg
2026-10-07 10:02:37 | INFO | __main__ | Собрано 8005 изображений из data/processed/images/val
2026-10-07 10:02:37 | INFO | __main__ | Сэмплировано 50 из 8005
2026-10-07 10:02:38 | INFO | __main__ | Отрисовано 50 изображений в data/samples/processed/val
2026-10-07 10:02:38 | INFO | __main__ | Метаданные: data/samples/processed/val/metadata.json
2026-10-07 10:02:38 | INFO | __main__ | Контактный лист: data/samples/processed/val/contact_sheet.jpg


In [14]:
!python -m src.collect_rejected --config configs/config.yaml

2026-10-07 10:02:45 | INFO | __main__ | Отвергнутых аннотаций: 2
2026-10-07 10:02:45 | INFO | __main__ | Сохранено 2 отвергнутых изображений в data/samples/rejected/drawn
2026-10-07 10:02:45 | INFO | __main__ | Смотреть: python -m src.viewer --dir data/samples/rejected/drawn


### 6.1. Просмотр сэмплов (слайдер)

Поменяйте `FOLDER`, чтобы посмотреть `train`, `val` или `rejected/drawn`.

In [17]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

FOLDER = Path("data/samples/rejected/drawn")  # ← обработанные: processed/train или processed/val
paths = sorted(FOLDER.glob("*.jpg"))

if not paths:
    raise RuntimeError(f"Нет изображений в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

IntSlider(value=0, description='0/1', max=1)

Output()

## 7. Обучение

Параметры (модель, эпохи, batch, аугментации) — в `configs/config.yaml`.

`--run-name` пишите осмысленно: так проще сопоставлять `runs/train/<run>` и `runs/val/<run>`.

In [18]:
!rm -rf runs/train runs/val
!python -m src.train --config configs/config.yaml --run-name smoke_yolov8s_30ep_no_mosaic

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
2026-10-07 10:04:08 | INFO | __main__ | Загрузка модели: yolov8s.pt
2026-10-07 10:04:09 | INFO | __main__ | Начало обучения: 30 эпох, batch=16, run_name='smoke_yolov8s_30ep_no_mosaic'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_

## 8. Оценка

Имя папки в `runs/val/` выводится автоматически из имени train-папки.

In [ ]:
!python -m src.evaluate --config configs/config.yaml

### 8.1. Кривые обучения

Слайдер по PNG-файлам в `runs/train/<run_name>/` (results, confusion_matrix, PR-кривые и т.п.).

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

train_runs = sorted(Path("runs/train").glob("*/"))
if not train_runs:
    raise RuntimeError("Нет прогонов в runs/train/")
FOLDER = train_runs[-1]  # последний прогон
paths = sorted(FOLDER.glob("*.png"))

if not paths:
    raise RuntimeError(f"Нет PNG в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

## 9. Сохранение результатов на Google Drive

Упаковываем `runs/` и `data/reports/` — чтобы после отключения Colab-рантайма остались метрики и логи.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!tar -czf /content/drive/MyDrive/rsna-runs.tar.gz \
     -C /content/rsna-pneumonia-yolo-pipeline runs data/reports data/staged

!ls -lh /content/drive/MyDrive/rsna-runs.tar.gz

## 10. Итоговая сводка по пайплайну

In [ ]:
import json
from pathlib import Path
import pandas as pd

def fmt(v):
    try:
        return f"{float(v):.4f}"
    except (TypeError, ValueError):
        return str(v)

print("=" * 60)
print("СВОДКА ПО ПАЙПЛАЙНУ")
print("=" * 60)

# 1. Отчёты по processed
for split in ("train", "val"):
    report_path = Path("data/reports") / f"processed_{split}" / "report.json"
    if report_path.exists():
        data = json.load(open(report_path))
        bpi = data.get("boxes_per_image", {})
        print(f"\n[processed/{split}]")
        print(f"  боксов:                 {data.get('num_boxes')}")
        print(f"  изображений с боксами:  {bpi.get('with_boxes')}")
        print(f"  изображений без боксов: {bpi.get('without_boxes')}")
        if "yolo_out_of_bounds" in data:
            print(f"  YOLO out-of-bounds:     {data['yolo_out_of_bounds']}")

# 2. Отчёт фильтрации
filt = Path("data/staged/filtering_report.json")
if filt.exists():
    data = json.load(open(filt))
    print(f"\n[Фильтрация]")
    for s in data.get("splits", []):
        print(f"  [{s['split']}]")
        print(f"    изображений всего:   {s['images_total']}")
        print(f"    positive:            {s['images_positive']}")
        print(f"    negative:            {s['images_negative']}")
        print(f"    dropped_to_rejected: {s['images_dropped_to_rejected']}")
        print(f"    боксов сохранено:    {s['boxes_kept']}")
        print(f"    боксов отвергнуто:   {s['boxes_rejected']}")
        for reason, count in s.get("rejected_by_reason", {}).items():
            if count:
                print(f"      {reason}: {count}")

# 3. Метрики последнего train-прогона
train_runs = sorted(Path("runs/train").glob("*/results.csv"))
if train_runs:
    last_run = train_runs[-1]
    df = pd.read_csv(last_run)
    last = df.iloc[-1]
    print(f"\n[Обучение — {last_run.parent.name}]")
    print(f"  эпох:         {int(last.get('epoch', -1))}")
    print(f"  mAP@0.5:      {fmt(last.get('metrics/mAP50(B)'))}")
    print(f"  mAP@0.5:0.95: {fmt(last.get('metrics/mAP50-95(B)'))}")
    print(f"  precision:    {fmt(last.get('metrics/precision(B)'))}")
    print(f"  recall:       {fmt(last.get('metrics/recall(B)'))}")

# 4. Прогоны оценки
val_runs = sorted(Path("runs/val").glob("*/"))
if val_runs:
    print(f"\n[Оценка]")
    for run in val_runs[-3:]:
        print(f"  {run.name}")

print("\n" + "=" * 60)